In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CanineModel, CanineTokenizer
import pandas as pd
from classes.style_encoder import StyleEncoder
from classes.conversational_dataset import ConversationDataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import string
import emoji

device = "cuda" if torch.cuda.is_available() else "cpu"

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1226 14:29:59.864000 25716 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
def author_contrastive_loss(z, authors, temperature=0.1):
    """
    z: [B, D] normalized embeddings
    authors: [B] author ids
    """
    z = F.normalize(z, dim=1)
    sim = z @ z.T / temperature  # [B, B]

    # mask self similarity
    mask = torch.eye(sim.size(0), device=sim.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # positives: same author
    pos_mask = authors.unsqueeze(0) == authors.unsqueeze(1)
    pos_mask = pos_mask & ~mask  # remove self

    # log-softmax over rows
    log_prob = F.log_softmax(sim, dim=1)

    # average over positives
    loss = -log_prob[pos_mask].mean()
    return loss

def compute_style_score(texts):
    scores = []
    for t in texts:
        if not isinstance(t, str):
            scores.append(0.0)
            continue

        emoji_count = sum(c in emoji.EMOJI_DATA for c in t)
        caps_ratio = sum(c.isupper() for c in t) / max(len(t), 1)
        punct_count = sum(c in string.punctuation for c in t)

        # simple scalar style intensity
        score = emoji_count + caps_ratio * 5 + punct_count * 0.5
        scores.append(score)

    return torch.tensor(scores, dtype=torch.float32)

def style_contrastive_loss(z_anchor, z_pos, z_neg, temperature=0.1):
    z_anchor = F.normalize(z_anchor, dim=1)
    z_pos = F.normalize(z_pos, dim=1)
    z_neg = F.normalize(z_neg, dim=1)

    pos_sim = torch.sum(z_anchor * z_pos, dim=1) / temperature
    neg_sim = torch.sum(z_anchor * z_neg, dim=1) / temperature

    logits = torch.stack([pos_sim, neg_sim], dim=1)
    labels = torch.zeros(z_anchor.size(0), dtype=torch.long, device=z_anchor.device)

    return F.cross_entropy(logits, labels)

def style_metric_loss(z_i, z_j, s_i, s_j):
    z_i = F.normalize(z_i, dim=1)
    z_j = F.normalize(z_j, dim=1)

    z_dist = 1 - torch.sum(z_i * z_j, dim=1)
    style_dist = torch.abs(s_i - s_j)

    return F.mse_loss(z_dist, style_dist)

In [3]:
rows = pd.read_csv("../data/train/retriever_train_undersampled.csv").to_dict(orient="records")

tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
dataset = ConversationDataset(rows, tokenizer)

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "author": torch.tensor([b["author"] for b in batch]),
        "content": [b["content"] for b in batch]
    }


loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True
)


In [ ]:
model = StyleEncoder().to(device)

optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.proj.parameters(), "lr": 5e-4}
])

epochs = 5
temperature = 0.1
lambda_metric = 0.5 # style contrastive only

model.train()

for epoch in range(epochs):
    total_loss = 0.0

    for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        authors = batch["author"].to(device)

        z = model(input_ids, attention_mask)

        loss = author_contrastive_loss(
            z=z,
            authors=authors,
            temperature=temperature
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

save_dir = "../models/style_retriever_author_contrastive"

model.eval()
model.cpu()  # move to CPU before saving

model.encoder.save_pretrained(save_dir)
torch.save(model.proj.state_dict(), f"{save_dir}/projection_head.pt")
tokenizer.save_pretrained(save_dir)


Epoch 1: 100%|██████████| 500/500 [03:50<00:00,  2.17it/s]


Epoch 1 | Loss: 3.1436


Epoch 2: 100%|██████████| 500/500 [04:24<00:00,  1.89it/s]


Epoch 2 | Loss: 2.9973


Epoch 3:   3%|▎         | 16/500 [00:09<04:42,  1.72it/s]

In [ ]:
# model = StyleEncoder().to(device)

# optimizer = torch.optim.AdamW([
#     {"params": model.encoder.parameters(), "lr": 1e-5},
#     {"params": model.proj.parameters(), "lr": 5e-4}
# ])

# epochs = 5
# temperature = 0.1
# lambda_metric = 0.5

# model.train()

# for epoch in range(epochs):
#     total_loss = 0

#     for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)

#         # raw text is required for style score
#         texts = batch["content"]


#         z = model(input_ids, attention_mask)

#         # ----- build positives / negatives -----
#         perm = torch.randperm(z.size(0))
#         z_pos = z[perm]

#         z_neg = torch.roll(z, shifts=1, dims=0)

#         # ----- style scores -----
#         style_scores = compute_style_score(texts).to(device)
#         s_i = style_scores
#         s_j = style_scores[perm]

#         # ----- losses -----
#         loss_contrastive = style_contrastive_loss(
#             z, z_pos, z_neg, temperature
#         )

#         loss_metric = style_metric_loss(
#             z, z_pos, s_i, s_j
#         )

#         loss = loss_contrastive + lambda_metric * loss_metric
        
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     avg_loss = total_loss / len(loader)
#     print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

# # -----------------------
# # Save model
# # -----------------------
# save_dir = "../models/style_retriever_style_contrastive"
# model.eval()
# model.cpu()

# model.encoder.save_pretrained(save_dir)
# torch.save(model.proj.state_dict(), f"{save_dir}/projection_head.pt")
# tokenizer.save_pretrained(save_dir)

Epoch 1:  48%|████▊     | 240/500 [01:49<01:58,  2.20it/s]


KeyboardInterrupt: 